# DefectVision — ResNet18 특징 추출

## 1. 구글 드라이브 연결

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. 데이터 압축 해제

In [ ]:
!mkdir -p /content/mvtec
!tar -xf /content/drive/MyDrive/bottle.tar.xz -C /content/mvtec
!ls /content/mvtec/bottle

ground_truth  license.txt  readme.txt  test  train


## 3. 데이터셋 로더 (전처리: 최민준)

In [ ]:
import os
import cv2
import torch
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split

class MVTecDataset(Dataset):
    def __init__(self, data_dir, category, split='train', img_size=(256, 256), val_ratio=0.1):
        self.data_dir = data_dir
        self.category = category
        self.split = split
        self.img_size = img_size
        self.image_paths = []
        self.labels = []
        self.mask_paths = []
        self._load_dataset(val_ratio)

    def _load_dataset(self, val_ratio):
        category_dir = os.path.join(self.data_dir, self.category)
        if self.split in ['train', 'val']:
            train_dir = os.path.join(category_dir, 'train', 'good')
            all_train_imgs = [os.path.join(train_dir, f) for f in os.listdir(train_dir) if f.endswith(('.png', '.jpg'))]
            train_imgs, val_imgs = train_test_split(all_train_imgs, test_size=val_ratio, random_state=42)
            self.image_paths = train_imgs if self.split == 'train' else val_imgs
            self.labels = [0] * len(self.image_paths)
            self.mask_paths = [None] * len(self.image_paths)
        elif self.split == 'test':
            test_dir = os.path.join(category_dir, 'test')
            gt_dir = os.path.join(category_dir, 'ground_truth')
            defect_types = os.listdir(test_dir)
            for defect in defect_types:
                img_dir = os.path.join(test_dir, defect)
                imgs = [f for f in os.listdir(img_dir) if f.endswith(('.png', '.jpg'))]
                for img_name in imgs:
                    self.image_paths.append(os.path.join(img_dir, img_name))
                    if defect == 'good':
                        self.labels.append(0)
                        self.mask_paths.append(None)
                    else:
                        self.labels.append(1)
                        mask_name = img_name.split('.')[0] + '_mask.png'
                        self.mask_paths.append(os.path.join(gt_dir, defect, mask_name))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_resized = cv2.resize(img, self.img_size)
        img_tensor = torch.from_numpy(img_resized).float().permute(2, 0, 1) / 255.0
        mask_tensor = torch.zeros((1, self.img_size[0], self.img_size[1]))
        if self.mask_paths[idx] is not None:
            mask = cv2.imread(self.mask_paths[idx], cv2.IMREAD_GRAYSCALE)
            mask_resized = cv2.resize(mask, self.img_size, interpolation=cv2.INTER_NEAREST)
            mask_tensor = torch.from_numpy(mask_resized).float().unsqueeze(0) / 255.0
        return img_tensor, self.labels[idx], mask_tensor, img_path

## 4. 데이터셋 확인

In [ ]:
DATA_DIR = '/content/mvtec'
CATEGORY = 'bottle'

train_ds = MVTecDataset(DATA_DIR, CATEGORY, split='train')
test_ds  = MVTecDataset(DATA_DIR, CATEGORY, split='test')
print('학습(정상) 장수:', len(train_ds), '/ 테스트 장수:', len(test_ds))

img, label, mask, path = train_ds[0]
print(img.shape)

학습(정상) 장수: 188 / 테스트 장수: 83
torch.Size([3, 256, 256])


## 5. ResNet18 불러오기

In [ ]:
import torchvision
model = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
model.eval()

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 135MB/s]


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

## 6. 특징 추출 함수

In [ ]:
import torch.nn.functional as F

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

def extract_features(x):
    x = (x - MEAN) / STD                 # ImageNet 정규화
    with torch.no_grad():                # 학습 안 함 → 빠르게
        x = model.conv1(x); x = model.bn1(x); x = model.relu(x); x = model.maxpool(x)
        f1 = model.layer1(x)             # 64채널
        f2 = model.layer2(f1)            # 128채널
        f3 = model.layer3(f2)            # 256채널 (layer4는 안 씀)
    size = f1.shape[-2:]
    f2 = F.interpolate(f2, size=size, mode='nearest')
    f3 = F.interpolate(f3, size=size, mode='nearest')
    return torch.cat([f1, f2, f3], dim=1)

print(extract_features(torch.rand(2, 3, 256, 256)).shape)

torch.Size([2, 448, 64, 64])


## 7. 특징 차원 축소

In [ ]:
D = 100
idx = torch.randperm(448, generator=torch.Generator().manual_seed(1024))[:D]

def get_embedding(x):
    return extract_features(x)[:, idx]   # (장수, 100, 64, 64)

## 8. 특징 추출 및 저장

In [ ]:
from torch.utils.data import DataLoader

def run(ds, save_name):
    loader = DataLoader(ds, batch_size=16, shuffle=False)
    embs, labels, masks, paths = [], [], [], []
    for img, label, mask, path in loader:
        embs.append(get_embedding(img))
        labels.append(label); masks.append(mask); paths += list(path)
    out = {'emb': torch.cat(embs), 'labels': torch.cat(labels),
           'masks': torch.cat(masks), 'paths': paths, 'idx': idx}
    torch.save(out, f'/content/drive/MyDrive/{save_name}')
    print(save_name, out['emb'].shape)

run(train_ds, f'{CATEGORY}_train_features.pt')
run(test_ds,  f'{CATEGORY}_test_features.pt')

bottle_train_features.pt torch.Size([188, 100, 64, 64])
bottle_test_features.pt torch.Size([83, 100, 64, 64])


## 9. CPU 처리 시간 측정

In [ ]:
import time, statistics

def benchmark(dim, res, warmup=5, repeat=30):
    x = torch.rand(1, 3, res, res)
    sel = idx[:dim] if dim <= D else torch.arange(448)
    for _ in range(warmup):
        extract_features(x)[:, sel]
    times = []
    for _ in range(repeat):
        t = time.perf_counter()
        extract_features(x)[:, sel]
        times.append((time.perf_counter() - t) * 1000)
    return statistics.mean(times), statistics.median(times)

print('CPU 스레드 수:', torch.get_num_threads())
for res in [256, 224]:
    for dim in [448, 100, 50]:
        m, med = benchmark(dim, res)
        print(f'해상도 {res} | 차원 {dim:3d} | 평균 {m:6.1f} ms | 중앙값 {med:6.1f} ms')

CPU 스레드 수: 1
해상도 256 | 차원 448 | 평균   97.2 ms | 중앙값   95.9 ms
해상도 256 | 차원 100 | 평균   95.6 ms | 중앙값   93.8 ms
해상도 256 | 차원  50 | 평균  104.4 ms | 중앙값   95.4 ms
해상도 224 | 차원 448 | 평균  108.0 ms | 중앙값  106.8 ms
해상도 224 | 차원 100 | 평균   69.7 ms | 중앙값   68.6 ms
해상도 224 | 차원  50 | 평균   69.8 ms | 중앙값   68.7 ms


## 10. 차원별 공분산 메모리 비교

In [ ]:
for d in [448, 100, 50]:
    mb = 64*64*d*d*4 / 1024**2
    print(f'차원 {d:3d}: 공분산 약 {mb:,.0f} MB')

차원 448: 공분산 약 3,136 MB
차원 100: 공분산 약 156 MB
차원  50: 공분산 약 39 MB
